<a href="https://colab.research.google.com/github/nanpolend/machine-learning/blob/master/_excel_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import streamlit as st
import pandas as pd
import numpy as np

# 網頁標題
st.title("生產良率與原料耗損率計算系統")

# 上傳檔案區
uploaded_file = st.file_uploader("上傳Excel檔案", type=["xlsx", "xls"])

if uploaded_file:
    try:
        # 讀取Excel檔案
        df = pd.read_excel(uploaded_file)

        # 顯示原始數據
        st.subheader("原始生產數據")
        st.dataframe(df)

        # 檢查必要欄位
        required_columns = ['原料代碼', '原料批號', '使用量', '良品數', '不良品數']
        if not all(col in df.columns for col in required_columns):
            missing = [col for col in required_columns if col not in df.columns]
            st.error(f"缺少必要欄位: {', '.join(missing)}")
            st.stop()

        # 計算產品良率（取全表平均值）
        total_good = df['良品數'].sum()
        total_bad = df['不良品數'].sum()
        yield_rate = total_good / (total_good + total_bad) * 100

        # 同原料加總計算
        material_summary = df.groupby('原料代碼').agg(
            總使用量=('使用量', 'sum'),
            標準用量=('標準用量', 'first')
        ).reset_index()

        # 計算原料耗損率（需要標準用量）
        if '標準用量' in df.columns:
            material_summary['耗損率(%)'] = round(
                (1 - (total_good * material_summary['標準用量'] / material_summary['總使用量'])) * 100, 2)
        else:
            st.warning("缺少'標準用量'欄位，無法計算原料耗損率")
            material_summary['耗損率(%)'] = np.nan

        # 顯示結果
        st.subheader("產品良率分析")
        st.metric("**產品總良率**", f"{yield_rate:.2f}%")

        st.subheader("原料耗損分析")
        st.dataframe(material_summary)

        # 顯示耗損率最高的原料
        if '標準用量' in df.columns:
            max_waste = material_summary.loc[material_summary['耗損率(%)'].idxmax()]
            st.warning(f"最高耗損原料: {max_waste['原料代碼']} (耗損率: {max_waste['耗損率(%)']}%)")

        # 下載按鈕
        st.download_button(
            label="下載分析結果",
            data=material_summary.to_csv(index=False).encode('utf-8-sig'),
            file_name='原料耗損分析.csv',
            mime='text/csv'
        )

    except Exception as e:
        st.error(f"處理錯誤: {str(e)}")
else:
    st.info("請上傳Excel檔案開始分析")

# 使用說明
st.subheader("使用說明")
st.markdown("""
1. **Excel格式要求**：
   - 必須包含欄位：`原料代碼`, `原料批號`, `使用量`, `良品數`, `不良品數`
   - 可選欄位：`標準用量`（計算原料耗損率所需）
   - 範例格式：

    | 原料代碼 | 原料批號 | 使用量 | 標準用量 | 良品數 | 不良品數 |
    |----------|----------|--------|----------|--------|----------|
    | A001     | B2023001 | 100    | 0.95     | 95     | 5        |
    | A001     | B2023002 | 150    | 0.95     | 140    | 10       |
    | B002     | C2023001 | 200    | 1.10     | 180    | 20       |

2. **計算公式**：
   - 產品良率 = 總良品數 / (總良品數 + 總不良品數) × 100%
   - 原料耗損率 = [1 - (良品數 × 標準用量) / 總使用量] × 100%
""")